In [28]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [ ]:
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

load_dotenv()
api_key = "pcsk_3PPYZV_9HMYBD6xUQecDWoWLCUEBtyyobMDPRQwW3hShxvpmbSXqLb8xjhYuwphPpwNCE9"

index_name = "hybrid-search-langchain-pinecone"

# initialise pinecone client
pc = Pinecone(api_key=api_key)

# create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # dimension of the dense model
        metric="dotproduct",  # sparse values supported only for dotproduct metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

# Compatibility shim for newer Pinecone SDKs with older LangChain retriever code.
# Patch the actual runtime instance method so positional calls work.
import inspect

_original_upsert = getattr(index, "upsert")


def compat_upsert(self, *args, **kwargs):
    # Accept positional `vectors` or keyword `vectors`.
    if "vectors" not in kwargs and len(args) > 0:
        kwargs["vectors"] = args[0]
    return _original_upsert(**kwargs)

# Bind the compatibility shim directly to this index instance.
index.upsert = compat_upsert.__get__(index, type(index))

print("Patched index.upsert", index.upsert)
print("upsert signature", inspect.signature(index.upsert))


Patched index.upsert <bound method compat_upsert of Index(host='https://hybrid-search-langchain-pinecone-hqtrpvc.svc.aped-4627-b74a.pinecone.io')>
upsert signature (vectors, namespace=None, **kwargs)


In [31]:
## vector embedding and sparse embedding models
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})
embeddings


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7805.40it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [32]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder=BM25Encoder().default()
bm25_encoder

In [33]:
sentences=[
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]

## tfidf values on these sentence
bm25_encoder.fit(sentences)

## store the values to a json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 234.38it/s]


In [ ]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x177c0bd50>, index=Index(host='https://hybrid-search-langchain-pinecone-hqtrpvc.svc.aped-4627-b74a.pinecone.io'))

In [ ]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"]
)

TypeError: PineconeHybridSearchRetriever.add_texts() missing 1 required positional argument: 'texts'